In [5]:
# =====================================================================
# COMPLETE INTERACTIVE DASHBOARD FOR FINANCIAL AGENT ANALYSIS
# =====================================================================
# This dashboard visualizes RL trading agent performance with:
# 1. Portfolio value vs benchmarks
# 2. Trading actions overlaid on price
# 3. PnL distribution analysis
# =====================================================================

# Import required libraries
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# =====================================================================
# 1. DATA GENERATION MODULE - FIXED VERSION
# =====================================================================
class TradingDataGenerator:
    """
    Generate realistic trading data with:
    - Market price series (simulated GBM)
    - RL Agent portfolio value
    - Benchmark portfolios (Buy & Hold, Random)
    - Trading actions (Buy/Sell/Hold)
    - PnL calculations
    """
    
    def __init__(self, n_periods=500, initial_price=100, initial_capital=10000):
        """
        Initialize data generator with realistic parameters.
        
        Parameters:
        -----------
        n_periods : int
            Number of time periods to simulate
        initial_price : float
            Starting price of the asset
        initial_capital : float
            Starting capital for all strategies
        """
        self.n_periods = n_periods
        self.initial_price = initial_price
        self.initial_capital = initial_capital
        self.risk_free_rate = 0.02 / 252  # Daily risk-free rate (2% annualized)
        
    def generate_price_series(self):
        """
        Generate realistic price series using Geometric Brownian Motion.
        
        Returns:
        --------
        np.array : Simulated price series
        """
        np.random.seed(42)  # For reproducibility
        
        dt = 1 / 252  # Daily frequency (252 trading days/year)
        mu = 0.08 / 252  # Daily drift (8% annualized)
        sigma = 0.2 / np.sqrt(252)  # Daily volatility (20% annualized)
        
        # GBM simulation
        price_series = np.zeros(self.n_periods)
        price_series[0] = self.initial_price
        
        for t in range(1, self.n_periods):
            # Random shock
            shock = np.random.normal(mu * dt, sigma * np.sqrt(dt))
            # Update price
            price_series[t] = price_series[t-1] * np.exp(shock)
            
        return price_series
    
    def generate_actions(self, price_series):
        """
        Generate realistic trading actions for RL agent.
        
        Parameters:
        -----------
        price_series : np.array
            Price series to base actions on
            
        Returns:
        --------
        tuple : (actions list, action_indices dict)
        """
        actions = ['Hold'] * self.n_periods
        action_indices = {'Buy': [], 'Sell': []}
        
        # Calculate simple returns
        if len(price_series) > 1:
            returns = np.diff(price_series) / price_series[:-1]
            
            for i in range(1, min(len(price_series)-1, len(returns))):
                # Simple moving averages
                short_window = 10
                long_window = 30
                
                if i >= long_window:
                    short_ma = np.mean(price_series[i-short_window:i])
                    long_ma = np.mean(price_series[i-long_window:i])
                    
                    # Buy when short MA crosses above long MA and price is rising
                    if short_ma > long_ma and returns[i-1] > 0:
                        actions[i] = 'Buy'
                        action_indices['Buy'].append(i)
                    # Sell when short MA crosses below long MA and price is falling
                    elif short_ma < long_ma and returns[i-1] < 0:
                        actions[i] = 'Sell'
                        action_indices['Sell'].append(i)
        
        return actions, action_indices
    
    def calculate_portfolio_values(self, price_series, actions):
        """
        Calculate portfolio values for different strategies.
        
        Parameters:
        -----------
        price_series : np.array
            Price series
        actions : list
            Trading actions
            
        Returns:
        --------
        dict : Portfolio values for each strategy
        """
        # Initialize portfolio trackers
        rl_portfolio = np.zeros(self.n_periods)
        buyhold_portfolio = np.zeros(self.n_periods)
        random_portfolio = np.zeros(self.n_periods)
        
        # RL Agent Strategy
        cash = self.initial_capital
        shares = 0
        rl_portfolio[0] = cash
        
        for i in range(1, self.n_periods):
            # RL Agent trading logic
            if actions[i] == 'Buy' and cash > price_series[i] * 10:
                shares_to_buy = min(10, cash // price_series[i])
                cash -= shares_to_buy * price_series[i]
                shares += shares_to_buy
            elif actions[i] == 'Sell' and shares > 0:
                shares_to_sell = min(5, shares)
                cash += shares_to_sell * price_series[i]
                shares -= shares_to_sell
            
            # Update portfolio value
            rl_portfolio[i] = cash + shares * price_series[i]
        
        # Buy & Hold Strategy
        bh_shares = self.initial_capital // price_series[0]
        bh_cash = self.initial_capital - bh_shares * price_series[0]
        buyhold_portfolio = bh_cash + bh_shares * price_series
        
        # Random Strategy (mean-reverting)
        np.random.seed(43)
        random_shares = 0
        random_cash = self.initial_capital
        random_portfolio[0] = random_cash
        
        for i in range(1, self.n_periods):
            if np.random.random() > 0.7:  # 30% chance to trade
                if np.random.random() > 0.5 and random_cash > price_series[i]:
                    # Buy
                    random_shares += 1
                    random_cash -= price_series[i]
                elif random_shares > 0:
                    # Sell
                    random_shares -= 1
                    random_cash += price_series[i]
            
            random_portfolio[i] = random_cash + random_shares * price_series[i]
        
        return {
            'RL_Agent': rl_portfolio,
            'Buy_Hold': buyhold_portfolio,
            'Random': random_portfolio
        }
    
    def calculate_pnl(self, portfolio_values, price_series, actions):
        """
        Calculate PnL metrics.
        
        Parameters:
        -----------
        portfolio_values : dict
            Portfolio values for each strategy
        price_series : np.array
            Price series
        actions : list
            Trading actions
            
        Returns:
        --------
        tuple : (incremental_pnl, cumulative_pnl, trade_pnl)
        """
        # Incremental PnL (daily) - ensure same length as portfolio values
        rl_values = portfolio_values['RL_Agent']
        incremental_pnl = np.diff(rl_values)  # This gives n-1 elements
        
        # Pad incremental PnL to match length
        incremental_pnl_padded = np.concatenate([[0], incremental_pnl])
        
        # Cumulative PnL - pad to match length
        cumulative_pnl = np.cumsum(incremental_pnl_padded)
        
        # Per-trade PnL
        trade_pnl = []
        in_trade = False
        entry_price = 0
        entry_index = 0
        
        for i, action in enumerate(actions):
            if action == 'Buy' and not in_trade:
                in_trade = True
                entry_price = price_series[i]
                entry_index = i
            elif action == 'Sell' and in_trade:
                in_trade = False
                trade_return = (price_series[i] - entry_price) / entry_price
                trade_pnl.append({
                    'entry_time': entry_index,
                    'exit_time': i,
                    'entry_price': entry_price,
                    'exit_price': price_series[i],
                    'pnl': trade_return,
                    'duration': i - entry_index
                })
        
        return incremental_pnl_padded, cumulative_pnl, trade_pnl
    
    def calculate_risk_metrics(self, portfolio_values):
        """
        Calculate Sharpe Ratio and Maximum Drawdown.
        
        Parameters:
        -----------
        portfolio_values : dict
            Portfolio values for each strategy
            
        Returns:
        --------
        dict : Risk metrics for each strategy
        """
        metrics = {}
        
        for strategy, values in portfolio_values.items():
            # Calculate returns
            returns = np.diff(values) / values[:-1]
            
            if len(returns) > 0 and np.std(returns) > 0:
                # Sharpe Ratio
                excess_returns = returns - self.risk_free_rate
                sharpe_ratio = np.sqrt(252) * np.mean(excess_returns) / np.std(excess_returns)
                
                # Maximum Drawdown
                cumulative_returns = np.cumprod(1 + returns)
                running_max = np.maximum.accumulate(cumulative_returns)
                drawdown = (cumulative_returns - running_max) / running_max
                max_drawdown = np.min(drawdown) if len(drawdown) > 0 else 0
                
                metrics[strategy] = {
                    'Sharpe_Ratio': sharpe_ratio,
                    'Max_Drawdown': max_drawdown,
                    'Annual_Return': np.mean(returns) * 252,
                    'Annual_Volatility': np.std(returns) * np.sqrt(252)
                }
            else:
                metrics[strategy] = {
                    'Sharpe_Ratio': 0,
                    'Max_Drawdown': 0,
                    'Annual_Return': 0,
                    'Annual_Volatility': 0
                }
        
        return metrics
    
    def generate_dataset(self):
        """
        Generate complete dataset for dashboard.
        
        Returns:
        --------
        pd.DataFrame : Complete trading dataset
        dict : Additional metrics and indices
        """
        # Generate price series
        price_series = self.generate_price_series()
        
        # Generate actions
        actions, action_indices = self.generate_actions(price_series)
        
        # Calculate portfolio values
        portfolio_values = self.calculate_portfolio_values(price_series, actions)
        
        # Calculate PnL
        incremental_pnl, cumulative_pnl, trade_pnl = self.calculate_pnl(
            portfolio_values, price_series, actions
        )
        
        # Calculate risk metrics
        risk_metrics = self.calculate_risk_metrics(portfolio_values)
        
        # Create timestamps
        start_date = datetime(2024, 1, 1)
        timestamps = [start_date + timedelta(days=i) for i in range(self.n_periods)]
        
        # Ensure all arrays have the same length
        assert len(timestamps) == len(price_series) == len(portfolio_values['RL_Agent']) == len(actions) == len(incremental_pnl) == len(cumulative_pnl), \
            f"Array length mismatch: timestamps={len(timestamps)}, prices={len(price_series)}, rl={len(portfolio_values['RL_Agent'])}, actions={len(actions)}, inc_pnl={len(incremental_pnl)}, cum_pnl={len(cumulative_pnl)}"
        
        # Create DataFrame
        data = {
            'Timestamp': timestamps,
            'Market_Price': price_series,
            'RL_Agent_Portfolio': portfolio_values['RL_Agent'],
            'Buy_Hold_Portfolio': portfolio_values['Buy_Hold'],
            'Random_Portfolio': portfolio_values['Random'],
            'Action': actions,
            'Incremental_PnL': incremental_pnl,
            'Cumulative_PnL': cumulative_pnl
        }
        
        df = pd.DataFrame(data)
        
        # Additional information for dashboard
        dashboard_data = {
            'trade_pnl': trade_pnl,
            'action_indices': action_indices,
            'risk_metrics': risk_metrics,
            'price_series': price_series,
            'portfolio_values': portfolio_values
        }
        
        return df, dashboard_data

# =====================================================================
# 2. DASHBOARD CREATION MODULE - FIXED VERSION
# =====================================================================
class TradingDashboard:
    """
    Create interactive dashboard with:
    1. Portfolio Value vs Benchmarks
    2. Actions Overlaid on Price
    3. PnL Distribution
    """
    
    def __init__(self, df, dashboard_data):
        """
        Initialize dashboard with data.
        
        Parameters:
        -----------
        df : pd.DataFrame
            Complete trading dataset
        dashboard_data : dict
            Additional metrics and indices
        """
        self.df = df
        self.dashboard_data = dashboard_data
        self.fig = None
        
    def create_dashboard(self):
        """
        Create complete interactive dashboard.
        
        Returns:
        --------
        go.Figure : Complete dashboard with all charts
        """
        # Create subplots with shared x-axis - SIMPLIFIED LAYOUT
        self.fig = make_subplots(
            rows=4, cols=2,
            subplot_titles=(
                '<b>Portfolio Value vs Benchmarks</b>',
                '<b>Performance Metrics</b>',
                '<b>Trading Actions Overlaid on Market Price</b>',
                '<b>Trade Statistics</b>',
                '<b>PnL Distribution per Trade</b>',
                '<b>Cumulative PnL</b>',
                '<b>Daily Returns Distribution</b>',
                '<b>Drawdown Analysis</b>'
            ),
            vertical_spacing=0.1,
            horizontal_spacing=0.15,
            specs=[
                [{"type": "scatter"}, {"type": "table"}],
                [{"type": "scatter"}, {"type": "table"}],
                [{"type": "histogram"}, {"type": "scatter"}],
                [{"type": "histogram"}, {"type": "scatter"}],
            ],
            row_heights=[0.25, 0.25, 0.25, 0.25]
        )
        
        # =============================================================
        # CHART 1: Portfolio Value vs Benchmarks
        # =============================================================
        portfolios = ['RL_Agent_Portfolio', 'Buy_Hold_Portfolio', 'Random_Portfolio']
        colors = ['#FF6B6B', '#4ECDC4', '#45B7D1']
        names = ['RL Agent', 'Buy & Hold', 'Random']
        
        for portfolio, color, name in zip(portfolios, colors, names):
            self.fig.add_trace(
                go.Scatter(
                    x=self.df['Timestamp'],
                    y=self.df[portfolio],
                    mode='lines',
                    name=name,
                    line=dict(color=color, width=2),
                    hovertemplate='<b>%{x|%Y-%m-%d}</b><br>' +
                                f'{name}: $%{{y:,.0f}}<br>' +
                                '<extra></extra>',
                    showlegend=True,
                    legendgroup='portfolios'
                ),
                row=1, col=1
            )
        
        # =============================================================
        # CHART 2: Performance Metrics Table
        # =============================================================
        risk_metrics = self.dashboard_data['risk_metrics']
        metrics_df = pd.DataFrame(risk_metrics).T
        metrics_df = metrics_df[['Sharpe_Ratio', 'Max_Drawdown', 'Annual_Return', 'Annual_Volatility']]
        metrics_df.columns = ['Sharpe Ratio', 'Max Drawdown', 'Ann Return', 'Ann Volatility']
        metrics_df = metrics_df.round(3)
        
        # Format for display
        metrics_df['Sharpe Ratio'] = metrics_df['Sharpe Ratio'].apply(lambda x: f"{x:.3f}")
        metrics_df['Max Drawdown'] = metrics_df['Max Drawdown'].apply(lambda x: f"{x:.2%}")
        metrics_df['Ann Return'] = metrics_df['Ann Return'].apply(lambda x: f"{x:.2%}")
        metrics_df['Ann Volatility'] = metrics_df['Ann Volatility'].apply(lambda x: f"{x:.2%}")
        
        self.fig.add_trace(
            go.Table(
                header=dict(
                    values=['<b>Strategy</b>'] + [f'<b>{col}</b>' for col in metrics_df.columns],
                    fill_color='#2C3E50',
                    align='center',
                    font=dict(color='white', size=11),
                    height=25
                ),
                cells=dict(
                    values=[metrics_df.index] + [metrics_df[col] for col in metrics_df.columns],
                    fill_color=['#F8F9FA', 'white'],
                    align='center',
                    font=dict(size=10),
                    height=22
                )
            ),
            row=1, col=2
        )
        
        # =============================================================
        # CHART 3: Actions Overlaid on Price
        # =============================================================
        # Price line
        self.fig.add_trace(
            go.Scatter(
                x=self.df['Timestamp'],
                y=self.df['Market_Price'],
                mode='lines',
                name='Price',
                line=dict(color='#2C3E50', width=2),
                hovertemplate='<b>%{x|%Y-%m-%d}</b><br>' +
                             'Price: $%{y:.2f}<br>' +
                             '<extra></extra>',
                showlegend=False
            ),
            row=2, col=1
        )
        
        # Buy actions
        buy_indices = self.dashboard_data['action_indices']['Buy']
        if buy_indices:
            buy_df = self.df.iloc[buy_indices]
            self.fig.add_trace(
                go.Scatter(
                    x=buy_df['Timestamp'],
                    y=buy_df['Market_Price'],
                    mode='markers',
                    name='Buy',
                    marker=dict(
                        symbol='triangle-up',
                        size=8,
                        color='#00C851',
                        line=dict(width=1, color='white')
                    ),
                    hovertemplate='<b>BUY</b><br>' +
                                 'Date: %{x|%Y-%m-%d}<br>' +
                                 'Price: $%{y:.2f}<br>' +
                                 '<extra></extra>',
                    showlegend=True,
                    legendgroup='actions'
                ),
                row=2, col=1
            )
        
        # Sell actions
        sell_indices = self.dashboard_data['action_indices']['Sell']
        if sell_indices:
            sell_df = self.df.iloc[sell_indices]
            self.fig.add_trace(
                go.Scatter(
                    x=sell_df['Timestamp'],
                    y=sell_df['Market_Price'],
                    mode='markers',
                    name='Sell',
                    marker=dict(
                        symbol='triangle-down',
                        size=8,
                        color='#ff4444',
                        line=dict(width=1, color='white')
                    ),
                    hovertemplate='<b>SELL</b><br>' +
                                 'Date: %{x|%Y-%m-%d}<br>' +
                                 'Price: $%{y:.2f}<br>' +
                                 '<extra></extra>',
                    showlegend=True,
                    legendgroup='actions'
                ),
                row=2, col=1
            )
        
        # Add moving average
        ma_period = 20
        self.df['MA_20'] = self.df['Market_Price'].rolling(window=ma_period).mean()
        self.fig.add_trace(
            go.Scatter(
                x=self.df['Timestamp'],
                y=self.df['MA_20'],
                mode='lines',
                name=f'{ma_period}-Day MA',
                line=dict(color='#FF8800', width=1, dash='dash'),
                opacity=0.7,
                showlegend=True,
                legendgroup='indicators'
            ),
            row=2, col=1
        )
        
        # =============================================================
        # CHART 4: Trade Statistics
        # =============================================================
        trade_pnl = self.dashboard_data['trade_pnl']
        if trade_pnl and len(trade_pnl) > 0:
            trade_stats = pd.DataFrame(trade_pnl)
            
            # Calculate statistics
            win_trades = trade_stats[trade_stats['pnl'] > 0]
            loss_trades = trade_stats[trade_stats['pnl'] < 0]
            
            stats_data = {
                'Metric': ['Total Trades', 'Win Rate', 'Profit Factor', 'Avg Win', 'Avg Loss', 'Avg Duration'],
                'Value': [
                    len(trade_pnl),
                    f"{(len(win_trades) / len(trade_pnl)):.1%}" if len(trade_pnl) > 0 else "0%",
                    f"{abs(win_trades['pnl'].sum() / loss_trades['pnl'].sum()):.2f}" if len(loss_trades) > 0 else "∞",
                    f"{win_trades['pnl'].mean():.2%}" if len(win_trades) > 0 else "0%",
                    f"{loss_trades['pnl'].mean():.2%}" if len(loss_trades) > 0 else "0%",
                    f"{trade_stats['duration'].mean():.0f} days" if len(trade_pnl) > 0 else "0 days"
                ]
            }
            
            self.fig.add_trace(
                go.Table(
                    header=dict(
                        values=['<b>Trade Metric</b>', '<b>Value</b>'],
                        fill_color='#2C3E50',
                        align='center',
                        font=dict(color='white', size=11),
                        height=25
                    ),
                    cells=dict(
                        values=[stats_data['Metric'], stats_data['Value']],
                        fill_color=['#F8F9FA', 'white'],
                        align='center',
                        font=dict(size=10),
                        height=22
                    )
                ),
                row=2, col=2
            )
        else:
            # Add empty table if no trades
            self.fig.add_trace(
                go.Table(
                    header=dict(
                        values=['<b>Trade Metric</b>', '<b>Value</b>'],
                        fill_color='#2C3E50',
                        align='center',
                        font=dict(color='white', size=11),
                        height=25
                    ),
                    cells=dict(
                        values=[['Total Trades', 'Win Rate', 'Profit Factor', 'Avg Win', 'Avg Loss', 'Avg Duration'], 
                               ['0', '0%', 'N/A', '0%', '0%', '0 days']],
                        fill_color=['#F8F9FA', 'white'],
                        align='center',
                        font=dict(size=10),
                        height=22
                    )
                ),
                row=2, col=2
            )
        
        # =============================================================
        # CHART 5: PnL Distribution
        # =============================================================
        if trade_pnl and len(trade_pnl) > 0:
            pnl_values = [trade['pnl'] for trade in trade_pnl]
            
            self.fig.add_trace(
                go.Histogram(
                    x=pnl_values,
                    nbinsx=15,
                    name='PnL Distribution',
                    marker_color='#33B5E5',
                    opacity=0.7,
                    hovertemplate='PnL: %{x:.2%}<br>' +
                                 'Count: %{y}<br>' +
                                 '<extra></extra>',
                    showlegend=False
                ),
                row=3, col=1
            )
            
            # Add Gaussian overlay if scipy is available
            try:
                from scipy.stats import norm
                if len(pnl_values) > 1:
                    x_range = np.linspace(min(pnl_values), max(pnl_values), 100)
                    mu, std = np.mean(pnl_values), np.std(pnl_values)
                    pdf = norm.pdf(x_range, mu, std) * len(pnl_values) * (max(pnl_values) - min(pnl_values)) / 15
                    
                    self.fig.add_trace(
                        go.Scatter(
                            x=x_range,
                            y=pdf,
                            mode='lines',
                            name='Normal Fit',
                            line=dict(color='#FF4444', width=2),
                            hovertemplate='PnL: %{x:.2%}<br>' +
                                         'Density: %{y:.2f}<br>' +
                                         '<extra></extra>',
                            showlegend=False
                        ),
                        row=3, col=1
                    )
            except ImportError:
                pass  # Silently skip if scipy is not installed
            
            # Add vertical line at zero
            self.fig.add_vline(x=0, line_dash="dash", line_color="gray", 
                              opacity=0.5, row=3, col=1)
        else:
            # Add empty histogram if no trades
            self.fig.add_trace(
                go.Histogram(
                    x=[0, 0.01, -0.01],  # Dummy data
                    name='PnL Distribution',
                    marker_color='#33B5E5',
                    opacity=0.7,
                    showlegend=False
                ),
                row=3, col=1
            )
        
        # =============================================================
        # CHART 6: Cumulative PnL
        # =============================================================
        self.fig.add_trace(
            go.Scatter(
                x=self.df['Timestamp'],
                y=self.df['Cumulative_PnL'],
                mode='lines',
                name='Cumulative PnL',
                line=dict(color='#AA66CC', width=2),
                fill='tozeroy',
                fillcolor='rgba(170, 102, 204, 0.2)',
                hovertemplate='<b>%{x|%Y-%m-%d}</b><br>' +
                             'PnL: $%{y:,.0f}<br>' +
                             '<extra></extra>',
                showlegend=False
            ),
            row=3, col=2
        )
        
        # Add horizontal line at zero
        self.fig.add_hline(y=0, line_dash="dash", line_color="gray", 
                          opacity=0.5, row=3, col=2)
        
        # =============================================================
        # CHART 7: Daily Returns Distribution
        # =============================================================
        # Calculate daily returns for RL Agent
        rl_returns = np.diff(self.df['RL_Agent_Portfolio']) / self.df['RL_Agent_Portfolio'].values[:-1]
        
        self.fig.add_trace(
            go.Histogram(
                x=rl_returns,
                nbinsx=30,
                name='Daily Returns',
                marker_color='#FF8800',
                opacity=0.7,
                hovertemplate='Return: %{x:.3%}<br>' +
                             'Count: %{y}<br>' +
                             '<extra></extra>',
                showlegend=False
            ),
            row=4, col=1
        )
        
        # Add vertical line at zero
        self.fig.add_vline(x=0, line_dash="dash", line_color="gray", 
                          opacity=0.5, row=4, col=1)
        
        # =============================================================
        # CHART 8: Drawdown Analysis
        # =============================================================
        # Calculate drawdown for RL Agent
        portfolio_values = self.df['RL_Agent_Portfolio'].values
        cumulative_returns = portfolio_values / portfolio_values[0]
        running_max = np.maximum.accumulate(cumulative_returns)
        drawdown = (cumulative_returns - running_max) / running_max
        
        self.fig.add_trace(
            go.Scatter(
                x=self.df['Timestamp'],
                y=drawdown * 100,  # Convert to percentage
                mode='lines',
                name='Drawdown',
                line=dict(color='#FF4444', width=2),
                fill='tozeroy',
                fillcolor='rgba(255, 68, 68, 0.2)',
                hovertemplate='<b>%{x|%Y-%m-%d}</b><br>' +
                             'Drawdown: %{y:.2f}%<br>' +
                             '<extra></extra>',
                showlegend=False
            ),
            row=4, col=2
        )
        
        # Add horizontal line at max drawdown
        max_dd = drawdown.min() * 100
        self.fig.add_hline(y=max_dd, line_dash="dash", line_color="darkred", 
                          opacity=0.7, row=4, col=2,
                          annotation_text=f"Max DD: {max_dd:.2f}%",
                          annotation_position="bottom right")
        
        # =============================================================
        # UPDATE LAYOUT
        # =============================================================
        self.fig.update_layout(
            title=dict(
                text='<b>TRADING AGENT PERFORMANCE DASHBOARD</b><br><span style="font-size:12px">Interactive Visualization of RL Trading Agent vs Benchmarks</span>',
                font=dict(size=22, color='#2C3E50'),
                x=0.5,
                xanchor='center',
                y=0.98
            ),
            height=1400,
            showlegend=True,
            hovermode='x unified',
            plot_bgcolor='rgba(240, 240, 240, 0.5)',
            paper_bgcolor='white',
            legend=dict(
                orientation="h",
                yanchor="bottom",
                y=1.02,
                xanchor="center",
                x=0.5,
                bgcolor='rgba(255, 255, 255, 0.8)',
                bordercolor='#2C3E50',
                borderwidth=1
            ),
            margin=dict(l=50, r=50, t=120, b=50)
        )
        
        # Update axes labels ONLY for charts that support them (not tables)
        # Row 1
        self.fig.update_xaxes(title_text="Date", row=1, col=1)
        self.fig.update_yaxes(title_text="Portfolio Value ($)", row=1, col=1)
        
        # Row 2
        self.fig.update_xaxes(title_text="Date", row=2, col=1)
        self.fig.update_yaxes(title_text="Price ($)", row=2, col=1)
        
        # Row 3
        self.fig.update_xaxes(title_text="PnL (%)", row=3, col=1)
        self.fig.update_yaxes(title_text="Frequency", row=3, col=1)
        self.fig.update_xaxes(title_text="Date", row=3, col=2)
        self.fig.update_yaxes(title_text="Cumulative PnL ($)", row=3, col=2)
        
        # Row 4
        self.fig.update_xaxes(title_text="Daily Return (%)", row=4, col=1)
        self.fig.update_yaxes(title_text="Frequency", row=4, col=1)
        self.fig.update_xaxes(title_text="Date", row=4, col=2)
        self.fig.update_yaxes(title_text="Drawdown (%)", row=4, col=2)
        
        # Update subplot titles
        for i in self.fig['layout']['annotations']:
            i['font'] = dict(size=12, color='#2C3E50')
        
        return self.fig
    
    def add_interactive_features(self):
        """
        Add interactive features to the dashboard.
        """
        if self.fig is None:
            self.create_dashboard()
        
        # Update menu for interactivity
        self.fig.update_layout(
            updatemenus=[
                dict(
                    type="buttons",
                    direction="left",
                    buttons=list([
                        dict(
                            args=[{"visible": [True, True, True]}],
                            label="Show All",
                            method="update"
                        ),
                        dict(
                            args=[{"visible": [True, False, False]}],
                            label="RL Agent Only",
                            method="update"
                        ),
                        dict(
                            args=[{"visible": [False, True, False]}],
                            label="Buy & Hold Only",
                            method="update"
                        ),
                        dict(
                            args=[{"visible": [False, False, True]}],
                            label="Random Only",
                            method="update"
                        )
                    ]),
                    pad={"r": 10, "t": 10},
                    showactive=True,
                    x=0.5,
                    xanchor="center",
                    y=1.15,
                    yanchor="top"
                )
            ]
        )
        
        # Add range slider for time filtering - ONLY on charts with time axes
        # Update the main x-axis (row1, col1) which affects all linked axes
        self.fig.update_xaxes(
            row=1, col=1,
            rangeselector=dict(
                buttons=list([
                    dict(count=1, label="1m", step="month", stepmode="backward"),
                    dict(count=6, label="6m", step="month", stepmode="backward"),
                    dict(count=1, label="YTD", step="year", stepmode="todate"),
                    dict(count=1, label="1y", step="year", stepmode="backward"),
                    dict(step="all")
                ]),
                bgcolor='rgba(44, 62, 80, 0.1)',
                bordercolor='#2C3E50',
                borderwidth=1,
                x=0,
                xanchor="left",
                y=1.1,
                yanchor="top"
            ),
            rangeslider=dict(visible=True),
            type="date"
        )
        
        # Link x-axes for synchronized zoom/pan (only for time-based charts)
        self.fig.update_xaxes(matches='x', row=1, col=1)
        self.fig.update_xaxes(matches='x', row=2, col=1)
        self.fig.update_xaxes(matches='x', row=3, col=2)
        self.fig.update_xaxes(matches='x', row=4, col=2)
        
        return self.fig

# =====================================================================
# 3. MAIN EXECUTION - FIXED VERSION
# =====================================================================
def main():
    """
    Main execution function to generate data and create dashboard.
    """
    print("=" * 70)
    print("TRADING AGENT DASHBOARD GENERATOR")
    print("=" * 70)
    print("\n1. Generating realistic trading data...")
    
    # Generate data
    generator = TradingDataGenerator(n_periods=500, initial_price=100, initial_capital=10000)
    df, dashboard_data = generator.generate_dataset()
    
    print(f"   • Generated {len(df)} trading periods")
    print(f"   • Buy actions: {len(dashboard_data['action_indices']['Buy'])}")
    print(f"   • Sell actions: {len(dashboard_data['action_indices']['Sell'])}")
    print(f"   • Completed trades: {len(dashboard_data['trade_pnl'])}")
    
    print("\n2. Calculating performance metrics...")
    risk_metrics = dashboard_data['risk_metrics']
    
    print("\n   Performance Metrics:")
    print("   " + "-" * 50)
    for strategy, metrics in risk_metrics.items():
        sharpe = metrics['Sharpe_Ratio']
        drawdown = metrics['Max_Drawdown']
        ann_return = metrics['Annual_Return']
        ann_vol = metrics['Annual_Volatility']
        print(f"   {strategy:15s} Sharpe: {sharpe:6.3f}  Max DD: {drawdown:7.2%}  Return: {ann_return:7.2%}  Vol: {ann_vol:7.2%}")
    
    print("\n3. Creating interactive dashboard...")
    
    # Create dashboard
    dashboard = TradingDashboard(df, dashboard_data)
    fig = dashboard.create_dashboard()
    fig = dashboard.add_interactive_features()
    
    print("   • Created 8 integrated visualizations")
    print("   • Added interactive controls (toggles, range slider)")
    print("   • Implemented synchronized hover and zoom")
    
    print("\n4. Displaying dashboard...")
    print("\n" + "=" * 70)
    print("DASHBOARD READY")
    print("=" * 70)
    print("\nDashboard Components:")
    print("  1. Portfolio Value vs Benchmarks")
    print("  2. Performance Metrics Table")
    print("  3. Trading Actions Overlaid on Price")
    print("  4. Trade Statistics Table")
    print("  5. PnL Distribution Histogram")
    print("  6. Cumulative PnL Tracking")
    print("  7. Daily Returns Distribution")
    print("  8. Drawdown Analysis")
    print("\nInteractive Features:")
    print("  ✓ Toggle visibility of strategies")
    print("  ✓ Time range selector (1m, 6m, YTD, 1y, All)")
    print("  ✓ Range slider for detailed time selection")
    print("  ✓ Synchronized hover across charts")
    print("  ✓ Pan and zoom capabilities")
    print("  ✓ Linked time axes for synchronized navigation")
    
    # Display the dashboard
    fig.show()
    
    # Return data for further analysis if needed
    return df, dashboard_data, fig

# =====================================================================
# 4. RUN THE DASHBOARD
# =====================================================================
if __name__ == "__main__":
    try:
        # Execute the dashboard creation
        df, dashboard_data, fig = main()
        
        # Optionally save the dashboard to HTML for sharing
        print("\n5. Saving dashboard to HTML file...")
        fig.write_html("trading_agent_dashboard.html", include_plotlyjs='cdn')
        print("   • Dashboard saved as 'trading_agent_dashboard.html'")
        print("   • Open in any web browser for interactive exploration")
        
        # Print data summary
        print("\n6. Data Summary:")
        print("   " + "-" * 50)
        print(f"   Dataset shape: {df.shape}")
        print(f"   Date range: {df['Timestamp'].min().date()} to {df['Timestamp'].max().date()}")
        print(f"   Initial capital: ${df['RL_Agent_Portfolio'].iloc[0]:,.0f}")
        print(f"   Final RL Agent portfolio: ${df['RL_Agent_Portfolio'].iloc[-1]:,.0f}")
        print(f"   Total cumulative PnL: ${df['Cumulative_PnL'].iloc[-1]:,.0f}")
        
    except Exception as e:
        print(f"\nError occurred: {type(e).__name__}: {e}")
        print("\nTroubleshooting steps:")
        print("1. Make sure required libraries are installed:")
        print("   pip install numpy pandas plotly")
        print("2. For Gaussian overlay (optional):")
        print("   pip install scipy")
        print("3. Run in Jupyter Notebook for best experience")
        print("\nIf scipy is not available, the code will run without Gaussian overlays.")

TRADING AGENT DASHBOARD GENERATOR

1. Generating realistic trading data...
   • Generated 500 trading periods
   • Buy actions: 117
   • Sell actions: 108
   • Completed trades: 9

2. Calculating performance metrics...

   Performance Metrics:
   --------------------------------------------------
   RL_Agent        Sharpe: -2.080  Max DD:  -0.84%  Return:   0.21%  Vol:   0.86%
   Buy_Hold        Sharpe: -1.433  Max DD:  -1.41%  Return:   0.23%  Vol:   1.23%
   Random          Sharpe: -10.222  Max DD:  -0.21%  Return:   0.10%  Vol:   0.19%

3. Creating interactive dashboard...

Error occurred: PlotlyKeyError: Invalid property specified for object of type plotly.graph_objs.Table: 'xaxis'

Did you mean "cells"?

    Valid properties:
        cells
            :class:`plotly.graph_objects.table.Cells` instance or
            dict with compatible properties
        columnorder
            Specifies the rendered order of the data columns; for
            example, a value `2` at position `0` 